
| Cell | Interface |
|------|-----------|
| 1 | **Setup** —> connect to database |
| 2 | **Frequent Shopper Sign Up** —> record a customer sale, update inventory |
| 3 | **Checkout Register** —> adds a new customer to the Customer table |
| 4 | **Online Order** —> place a web-store order |
| 5 | **Inventory Manager** —> add / remove / shift inventory between stores |
| 6 | **Restock / Order More** —> request warehouse replenishment |
| 7 | **OLAP Queries** —> run the manager reports |
| 8 | **Search** —> find any value across all tables |
| 9 | **Close Connection** |


## Cell 1 


In [1]:
import mysql.connector

cnx = mysql.connector.connect(
    user='root',
    password='Blue4Globe1!',  
    host='127.0.0.1',
    database='seafoodcity'
)
cursor = cnx.cursor()
print("Connected to seafoodcity database successfully.")

Connected to seafoodcity database successfully.


## Cell 2

Saves a customer into the Customer Database so they can get a Customer ID

In [8]:
print("=== Seafood City Frequent Shopper Sign Up ===\n")

full_name = input("Full name (required): ").strip()
if not full_name:
    print("Name is required to sign up.")
else:
    email = input("Email address (press Enter to skip): ").strip() or None
    address = input("Home address (press Enter to skip): ").strip() or None

    # Preview what will be saved
    print("\nReview your information:")
    print(f"  Name:    {full_name}")
    print(f"  Email:   {email if email else '(not provided)'}")
    print(f"  Address: {address if address else '(not provided)'}")

    confirm = input("\nSign up with this information? (yes/no): ").strip().lower()

    if confirm == 'yes':
        cursor.execute("""
            INSERT INTO Customer (FullName, CustomerEmail, Address)
            VALUES (%s, %s, %s)
        """, (full_name, email, address))
        cnx.commit()

        new_id = cursor.lastrowid
        print(f"\nWelcome to Seafood City, {full_name}!")
        print(f"Your Customer ID is: {new_id}")
        print("Please save this number — you will need it at checkout.")
    else:
        print("Sign-up cancelled.")

=== Seafood City Frequent Shopper Sign-Up ===



Full name (required):  Ellie Traan
Email address (press Enter to skip):  etraan@conncoll.edu
Home address (press Enter to skip):  270 Mohegan Avenue



Review your information:
  Name:    Ellie Traan
  Email:   etraan@conncoll.edu
  Address: 270 Mohegan Avenue



Sign up with this information? (yes/no):  yes



Welcome to Seafood City, Ellie Traan!
Your Customer ID is: 41
Please save this number — you will need it at checkout.


## Cell 3 
Records a customer's market basket, updates store inventory, and optionally links the sale to a frequent-shopper account.


In [3]:
def show_inventory(store_id):
    """Print available products and stock at a given store."""
    cursor.execute("""
        SELECT p.ProductID, p.Name, p.Brand, inv.Quantity, inv.UnitPrice
        FROM Inventory inv
        JOIN Products p ON inv.ProductID = p.ProductID
        WHERE inv.StoreID = %s
        ORDER BY p.ProductID
    """, (store_id,))
    rows = cursor.fetchall()
    print(f"\n{'ID':<6} {'Product Name':<45} {'Brand':<20} {'Stock':>6} {'Price':>8}")
    print("-" * 90)
    for pid, name, brand, qty, price in rows:
        print(f"{pid:<6} {name:<45} {str(brand):<20} {qty:>6} ${price:>7.2f}")

cursor.execute("""
    SELECT b.StoreID, b.Address, b.State
    FROM BrickAndMotor b ORDER BY b.StoreID
""")
print("Physical Stores:")
for sid, addr, state in cursor.fetchall():
    print(f"  Store {sid}: {addr}, {state}")

store_id = int(input("\nEnter Store ID: "))
show_inventory(store_id)

cust_input = input("\nEnter Customer ID (or press Enter to skip): ").strip()
customer_id = None
if cust_input:
    customer_id = int(cust_input)
    cursor.execute("SELECT FullName, CustomerEmail FROM Customer WHERE CustomerID = %s", (customer_id,))
    row = cursor.fetchone()
    if row:
        print(f"Welcome back, {row[0]} ({row[1]})!")
    else:
        print("Customer ID not found — proceeding as anonymous.")
        customer_id = None

pay_method = input("Payment method (Cash / Credit / Debit): ").strip().capitalize()

basket = []  
print("\nAdd items to basket. Type 'done' when finished.")
while True:
    pid_input = input("  Product ID (or 'done'): ").strip()
    if pid_input.lower() == 'done':
        break
    try:
        pid = int(pid_input)
    except ValueError:
        print("  Please enter a numeric Product ID.")
        continue

    cursor.execute("""
        SELECT p.Name, inv.Quantity, inv.UnitPrice
        FROM Inventory inv
        JOIN Products p ON inv.ProductID = p.ProductID
        WHERE inv.ProductID = %s AND inv.StoreID = %s
    """, (pid, store_id))
    item = cursor.fetchone()
    if not item:
        print(f"  Product {pid} is not stocked at this store.")
        continue

    name, stock, price = item
    qty = int(input(f"  Quantity for '{name}' (${price:.2f} each, {stock} in stock): "))
    if qty > stock:
        print(f"  Not enough stock — only {stock} available.")
        continue
    basket.append((pid, qty, price))
    print(f"  Added {qty}x {name} @ ${price:.2f}")

if not basket:
    print("No items in basket — transaction cancelled.")
else:
    total = sum(qty * price for _, qty, price in basket)
    print(f"\nBasket total: ${total:.2f}")
    confirm = input("Confirm sale? (yes/no): ").strip().lower()

    if confirm == 'yes':
        # Insert Transaction
        cursor.execute("""
            INSERT INTO `Transaction` (TotalPrice, TransactionDate)
            VALUES (%s, NOW())
        """, (total,))
        txn_id = cursor.lastrowid

        # Insert Processing (store → transaction)
        cursor.execute("""
            INSERT INTO Processing (StoreID, TransactionID) VALUES (%s, %s)
        """, (store_id, txn_id))

        # Insert Make (if frequent shopper)
        if customer_id:
            cursor.execute("""
                INSERT INTO Make (CustomerID, TransactionID, PayMethod)
                VALUES (%s, %s, %s)
            """, (customer_id, txn_id, pay_method))

        # Insert Included line items + update Inventory
        for pid, qty, price in basket:
            cursor.execute("""
                INSERT INTO Included (TransactionID, ProductID, QuantityPurchased, UnitPriceAtSale)
                VALUES (%s, %s, %s, %s)
            """, (txn_id, pid, qty, price))
            cursor.execute("""
                UPDATE Inventory SET Quantity = Quantity - %s
                WHERE ProductID = %s AND StoreID = %s
            """, (qty, pid, store_id))

        cnx.commit()
        print(f"\nSale recorded! Transaction ID: {txn_id} | Total: ${total:.2f}")
        print("Thank you for shopping at Seafood City!")
    else:
        print("Transaction cancelled.")

Physical Stores:
  Store 1: 1235 Highland Ave, National City, CA 91950, CA
  Store 2: 2700 N Garey Ave, Pomona, CA 91767, CA
  Store 3: 94-050 Farrington Hwy, Waipahu, HI 96797, HI
  Store 4: 3050 E Desert Inn Rd, Las Vegas, NV 89121, NV
  Store 5: 1122 S Western Ave, Los Angeles, CA 90006, CA
  Store 6: 6051 Mack Rd, Sacramento, CA 95823, CA
  Store 7: 4800 N Pulaski Rd, Chicago, IL 60630, IL



Enter Store ID:  3



ID     Product Name                                  Brand                 Stock    Price
------------------------------------------------------------------------------------------
1      Whole Tilapia (Fresh)                         Seafood City Fresh       80 $   5.49
2      Bangus (Milkfish) Whole                       Seafood City Fresh       75 $   5.99
3      Whole Pompano                                 Seafood City Fresh       55 $   8.79
5      Live Blue Crab                                Seafood City Fresh       50 $   9.99
6      Manila Clams                                  Seafood City Fresh       60 $   6.99
7      Jumbo Shrimp (Head-on)                        Seafood City Fresh       45 $  10.99
8      Squid (Pusit)                                 Seafood City Fresh       70 $   6.49
13     Pork Belly (Liempo) per lb                    Seafood City Fresh      140 $   3.79
17     Chicken Leg Quarters per lb                   Seafood City Fresh      180 $   1.69
27     F


Enter Customer ID (or press Enter to skip):  
Payment method (Cash / Credit / Debit):  Credit



Add items to basket. Type 'done' when finished.


  Product ID (or 'done'):  13
  Quantity for 'Pork Belly (Liempo) per lb' ($3.79 each, 140 in stock):  150


... (3 outputs left out)

  Product ID (or 'done'):  done



Basket total: $7.49


Confirm sale? (yes/no):  yes



Sale recorded! Transaction ID: 202 | Total: $7.49
Thank you for shopping at Seafood City!


## Cell 4 
Only online orders. Processes an order placed through the Seafood City website (StoreID = 8). Web customers must be registered frequent shoppers (have a CustomerID). Only products stocked in the web inventory (StoreID = 8) can be ordered.

In [5]:
WEB_STORE_ID = 8

cursor.execute("""
    SELECT p.ProductID, p.Name, p.Brand, inv.Quantity, inv.UnitPrice
    FROM Inventory inv
    JOIN Products p ON inv.ProductID = p.ProductID
    WHERE inv.StoreID = %s
    ORDER BY p.ProductID
""", (WEB_STORE_ID,))
rows = cursor.fetchall()
print("Web Store Catalog — seafoodcity.com")
print(f"{'ID':<6} {'Product Name':<45} {'Brand':<20} {'Stock':>6} {'Price':>8}")
print("-" * 90)
for pid, name, brand, qty, price in rows:
    print(f"{pid:<6} {name:<45} {str(brand):<20} {qty:>6} ${price:>7.2f}")

# Customer login — web store has NO anonymous customers
customer_id = int(input("\nEnter your Customer ID (required for web orders): "))
cursor.execute("SELECT FullName, CustomerEmail FROM Customer WHERE CustomerID = %s", (customer_id,))
row = cursor.fetchone()
if not row:
    print("Customer ID not found. Web orders require a registered account.")
else:
    print(f"Logged in as: {row[0]} ({row[1]})")
    pay_method = 'Online'

    # Build order
    basket = []
    print("\nAdd items to your order. Type 'done' when finished.")
    while True:
        pid_input = input("  Product ID (or 'done'): ").strip()
        if pid_input.lower() == 'done':
            break
        try:
            pid = int(pid_input)
        except ValueError:
            print("  Please enter a numeric Product ID.")
            continue
        cursor.execute("""
            SELECT p.Name, inv.Quantity, inv.UnitPrice
            FROM Inventory inv
            JOIN Products p ON inv.ProductID = p.ProductID
            WHERE inv.ProductID = %s AND inv.StoreID = %s
        """, (pid, WEB_STORE_ID))
        item = cursor.fetchone()
        if not item:
            print(f"  Product {pid} is not available online.")
            continue
        name, stock, price = item
        qty = int(input(f"  Quantity for '{name}' (${price:.2f} each, {stock} available): "))
        if qty > stock:
            print(f"  Not enough stock — only {stock} available online.")
            continue
        basket.append((pid, qty, price))
        print(f"  Added {qty}x {name} @ ${price:.2f}")

    if not basket:
        print("No items in order — cancelled.")
    else:
        total = sum(qty * price for _, qty, price in basket)
        print(f"\nOrder total: ${total:.2f}")
        confirm = input("Place order? (yes/no): ").strip().lower()

        if confirm == 'yes':
            cursor.execute("""
                INSERT INTO `Transaction` (TotalPrice, TransactionDate)
                VALUES (%s, NOW())
            """, (total,))
            txn_id = cursor.lastrowid

            cursor.execute("""
                INSERT INTO Processing (StoreID, TransactionID) VALUES (%s, %s)
            """, (WEB_STORE_ID, txn_id))

            cursor.execute("""
                INSERT INTO Make (CustomerID, TransactionID, PayMethod)
                VALUES (%s, %s, %s)
            """, (customer_id, txn_id, pay_method))

            for pid, qty, price in basket:
                cursor.execute("""
                    INSERT INTO Included (TransactionID, ProductID, QuantityPurchased, UnitPriceAtSale)
                    VALUES (%s, %s, %s, %s)
                """, (txn_id, pid, qty, price))
                cursor.execute("""
                    UPDATE Inventory SET Quantity = Quantity - %s
                    WHERE ProductID = %s AND StoreID = %s
                """, (qty, pid, WEB_STORE_ID))

            cnx.commit()
            print(f"\nOrder placed! Transaction ID: {txn_id} | Total: ${total:.2f}")
            print("Your order has been confirmed. Thank you for shopping at Seafood City!")
        else:
            print("Order cancelled.")

Web Store Catalog — seafoodcity.com
ID     Product Name                                  Brand                 Stock    Price
------------------------------------------------------------------------------------------
27     Frozen Tilapia Fillet 2lb                     Pamana Foods            500 $   8.49
28     Frozen Shrimp Peeled 1lb                      Pamana Foods            400 $   9.49
29     Frozen Lumpia Shanghai 50pc                   Marca Piña              349 $  10.49
30     Frozen Siopao (Asado) 6pc                     Pamana Foods            300 $   7.49
31     Frozen Halo-Halo Mix 16oz                     Selecta                 200 $   4.29
32     Selecta Ube Ice Cream 1.5L                    Selecta                 200 $   7.49
33     Selecta Mango Ice Cream 1.5L                  Selecta                 200 $   7.49
34     Datu Puti Vinegar 1L                          Datu Puti               300 $   2.79
35     Datu Puti Soy Sauce 1L                        Datu Puti 


Enter your Customer ID (required for web orders):  40


Logged in as: Felicisima Domingo (fdomingo@email.com)

Add items to your order. Type 'done' when finished.


  Product ID (or 'done'):  29
  Quantity for 'Frozen Lumpia Shanghai 50pc' ($10.49 each, 349 available):  2


  Added 2x Frozen Lumpia Shanghai 50pc @ $10.49


  Product ID (or 'done'):  41
  Quantity for 'Mang Tomas All-Purpose Sauce 12oz' ($2.79 each, 250 available):  1


  Added 1x Mang Tomas All-Purpose Sauce 12oz @ $2.79


  Product ID (or 'done'):  done



Order total: $23.77


Place order? (yes/no):  yes



Order placed! Transaction ID: 204 | Total: $23.77
Your order has been confirmed. Thank you for shopping at Seafood City!


## Cell 5 
- add= increase a product's stock at a store (or all stores)
- remove= decrease a product's stock at a store (or all stores)
- shift= move stock from one store to another

In [12]:
def print_product_inventory(product_id):
    """Show current stock of a product across all stores."""
    cursor.execute("""
        SELECT inv.StoreID,
               COALESCE(bm.Address, w.Url) AS Location,
               inv.Quantity
        FROM Inventory inv
        JOIN Store s ON inv.StoreID = s.StoreID
        LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
        LEFT JOIN Web w ON s.StoreID = w.StoreID
        WHERE inv.ProductID = %s
        ORDER BY inv.StoreID
    """, (product_id,))
    print(f"\n{'StoreID':<10} {'Location':<50} {'Stock':>6}")
    print("-" * 70)
    for sid, loc, qty in cursor.fetchall():
        print(f"{sid:<10} {str(loc):<50} {qty:>6}")

print("Inventory Manager — Operations: add, remove, or shift")
operation = input("Enter operation (add / remove / shift): ").strip().lower()

if operation not in ('add', 'remove', 'shift'):
    print("Invalid operation.")
else:
    product_id = int(input("Enter Product ID: "))

    cursor.execute("SELECT Name, Brand FROM Products WHERE ProductID = %s", (product_id,))
    prod = cursor.fetchone()
    if not prod:
        print("Product not found.")
    else:
        print(f"\nProduct: {prod[0]} ({prod[1]})")
        print_product_inventory(product_id)

        if operation in ('add', 'remove'):
            store_input = input("\nEnter Store ID (or 'all' for all stores): ").strip().lower()
            qty = int(input(f"Quantity to {operation}: "))

            if store_input == 'all':
                if operation == 'add':
                    cursor.execute("""
                        UPDATE Inventory SET Quantity = Quantity + %s
                        WHERE ProductID = %s
                    """, (qty, product_id))
                else: 
                    cursor.execute("""
                        UPDATE Inventory SET Quantity = GREATEST(Quantity - %s, 0)
                        WHERE ProductID = %s
                    """, (qty, product_id))
                cnx.commit()
                print(f"\n{operation.capitalize()}ed {qty} units at ALL stores for product {product_id}.")
            else:
                store_id = int(store_input)
                if operation == 'add':
                    cursor.execute("""
                        UPDATE Inventory SET Quantity = Quantity + %s
                        WHERE ProductID = %s AND StoreID = %s
                    """, (qty, product_id, store_id))
                else: 
                    cursor.execute("""
                        SELECT Quantity FROM Inventory
                        WHERE ProductID = %s AND StoreID = %s
                    """, (product_id, store_id))
                    current = cursor.fetchone()
                    if not current:
                        print("That product is not stocked at that store.")
                        qty = None
                    elif qty > current[0]:
                        print(f"Cannot remove {qty} — only {current[0]} in stock. Removing all available.")
                        qty = current[0]
                    if qty is not None:
                        cursor.execute("""
                            UPDATE Inventory SET Quantity = Quantity - %s
                            WHERE ProductID = %s AND StoreID = %s
                        """, (qty, product_id, store_id))
                if qty is not None:
                    cnx.commit()
                    print(f"\n{operation.capitalize()}ed {qty} units at Store {store_id} for product {product_id}.")

        elif operation == 'shift':
            from_store = int(input("\nShift FROM Store ID: "))
            to_store   = int(input("Shift TO Store ID: "))
            qty        = int(input("Quantity to shift: "))

            cursor.execute("""
                SELECT Quantity FROM Inventory
                WHERE ProductID = %s AND StoreID = %s
            """, (product_id, from_store))
            src = cursor.fetchone()
            if not src:
                print("Source store does not stock this product.")
            elif qty > src[0]:
                print(f"Not enough stock at Store {from_store} — only {src[0]} available.")
            else:
                cursor.execute("""
                    UPDATE Inventory SET Quantity = Quantity - %s
                    WHERE ProductID = %s AND StoreID = %s
                """, (qty, product_id, from_store))

                cursor.execute("""
                    SELECT Quantity, UnitPrice FROM Inventory
                    WHERE ProductID = %s AND StoreID = %s
                """, (product_id, to_store))
                dest = cursor.fetchone()

                if dest:
                    # Row already exists — just add to it
                    cursor.execute("""
                        UPDATE Inventory SET Quantity = Quantity + %s
                        WHERE ProductID = %s AND StoreID = %s
                    """, (qty, product_id, to_store))
                else:
                    # New row — get the price from the source store and insert
                    cursor.execute("""
                        SELECT UnitPrice FROM Inventory
                        WHERE ProductID = %s AND StoreID = %s
                    """, (product_id, from_store))
                    unit_price = cursor.fetchone()[0]
                    cursor.execute("""
                        INSERT INTO Inventory (ProductID, StoreID, Quantity, UnitPrice)
                        VALUES (%s, %s, %s, %s)
                    """, (product_id, to_store, qty, unit_price))


                cnx.commit()
                print(f"\nShifted {qty} units of product {product_id} from Store {from_store} to Store {to_store}.")

        print("\nUpdated inventory:")
        print_product_inventory(product_id)

Inventory Manager — Operations: add, remove, or shift


Enter operation (add / remove / shift):  shift
Enter Product ID:  23



Product: Jackfruit (Langka) per lb (Local Farm)

StoreID    Location                                            Stock
----------------------------------------------------------------------
1          1235 Highland Ave, National City, CA 91950             60
5          1122 S Western Ave, Los Angeles, CA 90006              50



Shift FROM Store ID:  1
Shift TO Store ID:  5
Quantity to shift:  1



Shifted 1 units of product 23 from Store 1 to Store 5.

Updated inventory:

StoreID    Location                                            Stock
----------------------------------------------------------------------
1          1235 Highland Ave, National City, CA 91950             59
5          1122 S Western Ave, Los Angeles, CA 90006              51


## Cell 6
Records a warehouse-to-store restock event and updates the store's inventory. You can also view the current warehouse stock levels.

In [13]:
cursor.execute("SELECT WarehouseID, Location, Region, Inventory FROM Warehouse ORDER BY WarehouseID")
print("Available Warehouses:")
print(f"{'ID':<5} {'Location':<50} {'Region':<25} {'Total Stock':>12}")
print("-" * 95)
for wid, loc, region, inv in cursor.fetchall():
    print(f"{wid:<5} {loc:<50} {str(region):<25} {inv:>12}")

warehouse_id = int(input("\nEnter Warehouse ID to ship from: "))

# Show products available at that warehouse
cursor.execute("""
    SELECT st.ProductID, p.Name, p.Brand, st.StockQuantity
    FROM Stock st
    JOIN Products p ON st.ProductID = p.ProductID
    WHERE st.WarehouseID = %s
    ORDER BY st.ProductID
""", (warehouse_id,))
wh_stock = cursor.fetchall()
print(f"\nWarehouse {warehouse_id} — Available Products:")
print(f"{'ID':<6} {'Product':<45} {'Brand':<20} {'Warehouse Stock':>15}")
print("-" * 90)
for pid, name, brand, qty in wh_stock:
    print(f"{pid:<6} {name:<45} {str(brand):<20} {qty:>15}")

# Show stores
cursor.execute("SELECT StoreID, Address, State FROM BrickAndMotor ORDER BY StoreID")
print("\nPhysical Stores:")
for sid, addr, state in cursor.fetchall():
    print(f"  Store {sid}: {addr}, {state}")

store_id   = int(input("\nEnter Store ID to restock: "))
product_id = int(input("Enter Product ID to restock: "))
qty        = int(input("Enter quantity to ship: "))

# Check warehouse has enough
cursor.execute("""
    SELECT StockQuantity FROM Stock
    WHERE WarehouseID = %s AND ProductID = %s
""", (warehouse_id, product_id))
wh_row = cursor.fetchone()

if not wh_row:
    print("That product is not available at that warehouse.")
elif qty > wh_row[0]:
    print(f"Not enough warehouse stock — only {wh_row[0]} available.")
else:
    import datetime
    restock_date = datetime.date.today()

    # Insert Restock record
    cursor.execute("""
        INSERT INTO Restock (WarehouseID, StoreID, ProductID, Restock_Date, Restock_Quantity)
        VALUES (%s, %s, %s, %s, %s)
    """, (warehouse_id, store_id, product_id, restock_date, qty))

    # Decrease warehouse stock
    cursor.execute("""
        UPDATE Stock SET StockQuantity = StockQuantity - %s
        WHERE WarehouseID = %s AND ProductID = %s
    """, (qty, warehouse_id, product_id))

    # Increase store inventory (INSERT if not stocked yet)
    cursor.execute("""
        SELECT UnitPrice FROM Inventory
        WHERE ProductID = %s AND StoreID = %s
    """, (product_id, store_id))
    existing = cursor.fetchone()
    if existing:
        cursor.execute("""
            UPDATE Inventory SET Quantity = Quantity + %s
            WHERE ProductID = %s AND StoreID = %s
        """, (qty, product_id, store_id))
    else:
        # Use base product price for new store entry
        cursor.execute("SELECT UnitPrice FROM Products WHERE ProductID = %s", (product_id,))
        base_price = cursor.fetchone()[0]
        cursor.execute("""
            INSERT INTO Inventory (ProductID, StoreID, Quantity, UnitPrice)
            VALUES (%s, %s, %s, %s)
        """, (product_id, store_id, qty, base_price))

    cnx.commit()
    print(f"\nRestock recorded! {qty} units of product {product_id} sent to Store {store_id} from Warehouse {warehouse_id}.")
    print(f"Restock date: {restock_date}")

Available Warehouses:
ID    Location                                           Region                     Total Stock
-----------------------------------------------------------------------------------------------
1     3800 E Inland Empire Blvd, Pomona, CA              Southern California              38000
2     1500 N Brea Blvd, Fullerton, CA                    Orange County                    22000
3     4200 E Olympic Blvd, Los Angeles, CA               Greater Los Angeles              15000
4     8000 Miramar Rd, San Diego, CA                     San Diego                        11000



Enter Warehouse ID to ship from:  1



Warehouse 1 — Available Products:
ID     Product                                       Brand                Warehouse Stock
------------------------------------------------------------------------------------------
1      Whole Tilapia (Fresh)                         Seafood City Fresh              2000
2      Bangus (Milkfish) Whole                       Seafood City Fresh              1800
3      Whole Pompano                                 Seafood City Fresh              1200
4      Whole Catfish                                 Seafood City Fresh              1500
5      Live Blue Crab                                Seafood City Fresh               800
6      Manila Clams                                  Seafood City Fresh               900
7      Jumbo Shrimp (Head-on)                        Seafood City Fresh               700
8      Squid (Pusit)                                 Seafood City Fresh              1100
9      Dried Dilis (Anchovies) 8oz                   Barrio Fies


Enter Store ID to restock:  6
Enter Product ID to restock:  60
Enter quantity to ship:  100



Restock recorded! 100 units of product 60 sent to Store 6 from Warehouse 1.
Restock date: 2026-04-27


## Cell 7 
OLAP Manager Reports. Run any of the five required project queries. Enter the query number when prompted.

In [14]:
# ── OLAP MANAGER REPORTS ──────────────────────────────────────

print("Manager Reports")
print("  1 — Top 20 best-selling products at each store")
print("  2 — Top 20 best-selling products in each state")
print("  3 — Top 5 stores with most sales this year")
print("  4 — Stores where Pork Belly outsells Chicken (Coke vs Pepsi equivalent)")
print("  5 — Top 3 categories bought together with Rice (like 'bought with milk')")
print("  6 — Monthly sales breakdown by store")
print("  7 — Average transaction value per store")
print("  8 — Customer purchase history (enter a Customer ID)")

choice = input("\nEnter report number (1-8): ").strip()

# ── Report 1 ──────────────────────────────────────────────────
if choice == '1':
    cursor.execute("""
        WITH StoreSales AS (
            SELECT pr.StoreID,
                   COALESCE(bm.Address, w.Url) AS StoreLocation,
                   p.ProductID, p.Name AS ProductName, p.Brand,
                   SUM(i.QuantityPurchased) AS TotalUnitsSold,
                   SUM(i.QuantityPurchased * i.UnitPriceAtSale) AS TotalRevenue,
                   RANK() OVER (
                       PARTITION BY pr.StoreID
                       ORDER BY SUM(i.QuantityPurchased) DESC
                   ) AS RankAtStore
            FROM Included i
            JOIN `Transaction` t ON i.TransactionID = t.TransactionID
            JOIN Processing pr   ON t.TransactionID = pr.TransactionID
            JOIN Store s         ON pr.StoreID = s.StoreID
            LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
            LEFT JOIN Web w            ON s.StoreID = w.StoreID
            JOIN Products p ON i.ProductID = p.ProductID
            GROUP BY pr.StoreID, COALESCE(bm.Address, w.Url),
                     p.ProductID, p.Name, p.Brand
        )
        SELECT * FROM StoreSales WHERE RankAtStore <= 20
        ORDER BY StoreID, RankAtStore
    """)
    rows = cursor.fetchall()
    print(f"\n{'Store':<7} {'Location':<40} {'Rank':<6} {'Product':<40} {'Units':>6} {'Revenue':>10}")
    print("-" * 115)
    for sid, loc, pid, name, brand, units, rev, rank in rows:
        print(f"{sid:<7} {str(loc)[:38]:<40} {rank:<6} {name[:38]:<40} {units:>6} ${rev:>9.2f}")

# ── Report 2 ──────────────────────────────────────────────────
elif choice == '2':
    cursor.execute("""
        WITH StateSales AS (
            SELECT bm.State, p.ProductID, p.Name AS ProductName, p.Brand,
                   SUM(i.QuantityPurchased) AS TotalUnitsSold,
                   SUM(i.QuantityPurchased * i.UnitPriceAtSale) AS TotalRevenue,
                   RANK() OVER (
                       PARTITION BY bm.State
                       ORDER BY SUM(i.QuantityPurchased) DESC
                   ) AS RankInState
            FROM Included i
            JOIN `Transaction` t  ON i.TransactionID = t.TransactionID
            JOIN Processing pr    ON t.TransactionID = pr.TransactionID
            JOIN BrickAndMotor bm ON pr.StoreID = bm.StoreID
            JOIN Products p       ON i.ProductID = p.ProductID
            GROUP BY bm.State, p.ProductID, p.Name, p.Brand
        )
        SELECT * FROM StateSales WHERE RankInState <= 20
        ORDER BY State, RankInState
    """)
    rows = cursor.fetchall()
    print(f"\n{'State':<7} {'Rank':<6} {'Product':<45} {'Brand':<20} {'Units':>6} {'Revenue':>10}")
    print("-" * 100)
    for state, pid, name, brand, units, rev, rank in rows:
        print(f"{state:<7} {rank:<6} {name[:43]:<45} {str(brand)[:18]:<20} {units:>6} ${rev:>9.2f}")

# ── Report 3 ──────────────────────────────────────────────────
elif choice == '3':
    cursor.execute("""
        WITH YearlySales AS (
            SELECT s.StoreID,
                   COALESCE(bm.Address, w.Url) AS StoreLocation,
                   COALESCE(bm.State, 'WEB') AS State,
                   COUNT(DISTINCT t.TransactionID) AS NumberOfTransactions,
                   SUM(i.QuantityPurchased * i.UnitPriceAtSale) AS TotalSalesRevenue,
                   RANK() OVER (
                       ORDER BY SUM(i.QuantityPurchased * i.UnitPriceAtSale) DESC
                   ) AS SalesRank
            FROM `Transaction` t
            JOIN Processing pr         ON t.TransactionID = pr.TransactionID
            JOIN Store s               ON pr.StoreID = s.StoreID
            LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
            LEFT JOIN Web w            ON s.StoreID = w.StoreID
            JOIN Included i            ON t.TransactionID = i.TransactionID
            WHERE YEAR(t.TransactionDate) = (
                SELECT MAX(YEAR(TransactionDate)) FROM `Transaction`
            )
            GROUP BY s.StoreID,
                     COALESCE(bm.Address, w.Url),
                     COALESCE(bm.State, 'WEB')
        )
        SELECT * FROM YearlySales ORDER BY TotalSalesRevenue DESC LIMIT 5
    """)
    rows = cursor.fetchall()
    print(f"\nTop 5 Stores by Sales Revenue (Most Recent Year)")
    print(f"{'Rank':<6} {'Store':<7} {'Location':<45} {'State':<7} {'Txns':>5} {'Revenue':>12}")
    print("-" * 90)
    for sid, loc, state, txns, rev, rank in rows:
        print(f"{rank:<6} {sid:<7} {str(loc)[:43]:<45} {state:<7} {txns:>5} ${rev:>11.2f}")

# ── Report 4 ──────────────────────────────────────────────────
elif choice == '4':
    cursor.execute("""
        WITH StoreMeatSales AS (
            SELECT pr.StoreID, p.Name AS MeatName,
                   SUM(i.QuantityPurchased) AS UnitsSold
            FROM Included i
            JOIN `Transaction` t ON i.TransactionID = t.TransactionID
            JOIN Processing pr   ON t.TransactionID = pr.TransactionID
            JOIN Products p      ON i.ProductID = p.ProductID
            WHERE p.Name IN ('Pork Belly (Liempo) per lb','Chicken Leg Quarters per lb')
            GROUP BY pr.StoreID, p.Name
        ),
        PorkSales    AS (SELECT StoreID, UnitsSold FROM StoreMeatSales WHERE MeatName = 'Pork Belly (Liempo) per lb'),
        ChickenSales AS (SELECT StoreID, UnitsSold FROM StoreMeatSales WHERE MeatName = 'Chicken Leg Quarters per lb')
        SELECT COUNT(*) AS StoresPorkBeatChicken,
               GROUP_CONCAT(pk.StoreID ORDER BY pk.StoreID) AS StoreIDs
        FROM PorkSales pk
        JOIN ChickenSales ck ON pk.StoreID = ck.StoreID
        WHERE pk.UnitsSold > ck.UnitsSold
    """)
    row = cursor.fetchone()
    print(f"\nStores where Pork Belly outsells Chicken Leg Quarters: {row[0]}")
    print(f"Store IDs: {row[1]}")

# ── Report 5 ──────────────────────────────────────────────────
elif choice == '5':
    cursor.execute("""
        WITH RiceBuyers AS (
            SELECT DISTINCT i.TransactionID
            FROM Included i
            JOIN Products p ON i.ProductID = p.ProductID
            WHERE p.Name IN ('Kokuho Rose Calrose Rice 15lb','Dinorado Premium Rice 25lb')
        ),
        CoProducts AS (
            SELECT c.CategoryName,
                   SUM(i.QuantityPurchased) AS TotalUnitsBoughtWithRice
            FROM RiceBuyers rb
            JOIN Included i   ON rb.TransactionID = i.TransactionID
            JOIN Belongs b    ON i.ProductID = b.ProductID
            JOIN Category c   ON b.CatID = c.CatID
            JOIN Products p   ON i.ProductID = p.ProductID
            WHERE p.Name NOT IN ('Kokuho Rose Calrose Rice 15lb','Dinorado Premium Rice 25lb')
            GROUP BY c.CategoryName
        )
        SELECT CategoryName, TotalUnitsBoughtWithRice,
               RANK() OVER (ORDER BY TotalUnitsBoughtWithRice DESC) AS CategoryRank
        FROM CoProducts
        ORDER BY TotalUnitsBoughtWithRice DESC
        LIMIT 3
    """)
    rows = cursor.fetchall()
    print(f"\nTop 3 categories bought together with Rice:")
    print(f"{'Rank':<6} {'Category':<30} {'Units Bought With Rice':>22}")
    print("-" * 62)
    for name, units, rank in rows:
        print(f"{rank:<6} {name:<30} {units:>22}")

# ── Report 6 ──────────────────────────────────────────────────
elif choice == '6':
    cursor.execute("""
        SELECT s.StoreID,
               COALESCE(bm.Address, w.Url) AS StoreLocation,
               DATE_FORMAT(t.TransactionDate, '%Y-%m') AS YearMonth,
               COUNT(DISTINCT t.TransactionID) AS NumberOfTransactions,
               SUM(i.QuantityPurchased) AS TotalItemsSold,
               SUM(i.QuantityPurchased * i.UnitPriceAtSale) AS MonthlyRevenue
        FROM `Transaction` t
        JOIN Processing pr         ON t.TransactionID = pr.TransactionID
        JOIN Store s               ON pr.StoreID = s.StoreID
        LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
        LEFT JOIN Web w            ON s.StoreID = w.StoreID
        JOIN Included i            ON t.TransactionID = i.TransactionID
        GROUP BY s.StoreID, COALESCE(bm.Address, w.Url),
                 DATE_FORMAT(t.TransactionDate, '%Y-%m')
        ORDER BY s.StoreID, YearMonth
    """)
    rows = cursor.fetchall()
    print(f"\n{'Store':<7} {'Month':<9} {'Txns':>5} {'Items':>6} {'Revenue':>12}  Location")
    print("-" * 100)
    for sid, loc, month, txns, items, rev in rows:
        print(f"{sid:<7} {month:<9} {txns:>5} {items:>6} ${rev:>11.2f}  {str(loc)[:40]}")

# ── Report 7 ──────────────────────────────────────────────────
elif choice == '7':
    cursor.execute("""
        SELECT s.StoreID,
               COALESCE(bm.Address, w.Url) AS StoreLocation,
               COALESCE(bm.State, 'WEB') AS State,
               COUNT(DISTINCT t.TransactionID) AS TotalTransactions,
               ROUND(SUM(i.QuantityPurchased * i.UnitPriceAtSale) /
                     COUNT(DISTINCT t.TransactionID), 2) AS AvgTransactionValue,
               SUM(i.QuantityPurchased * i.UnitPriceAtSale) AS TotalRevenue
        FROM `Transaction` t
        JOIN Processing pr         ON t.TransactionID = pr.TransactionID
        JOIN Store s               ON pr.StoreID = s.StoreID
        LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
        LEFT JOIN Web w            ON s.StoreID = w.StoreID
        JOIN Included i            ON t.TransactionID = i.TransactionID
        GROUP BY s.StoreID, COALESCE(bm.Address, w.Url), COALESCE(bm.State, 'WEB')
        ORDER BY AvgTransactionValue DESC
    """)
    rows = cursor.fetchall()
    print(f"\n{'Store':<7} {'State':<7} {'Txns':>5} {'Avg Txn Value':>14} {'Total Revenue':>14}  Location")
    print("-" * 105)
    for sid, loc, state, txns, avg, total in rows:
        print(f"{sid:<7} {state:<7} {txns:>5} ${avg:>13.2f} ${total:>13.2f}  {str(loc)[:40]}")

# ── Report 8 ──────────────────────────────────────────────────
elif choice == '8':
    cid = int(input("Enter Customer ID: "))
    cursor.execute("""
        SELECT c.CustomerID, c.FullName, t.TransactionID, t.TransactionDate,
               mk.PayMethod, COALESCE(bm.Address, w.Url) AS StoreLocation,
               p.Name, i.QuantityPurchased, i.UnitPriceAtSale,
               (i.QuantityPurchased * i.UnitPriceAtSale) AS LineTotal
        FROM Customer c
        JOIN Make mk               ON c.CustomerID = mk.CustomerID
        JOIN `Transaction` t       ON mk.TransactionID = t.TransactionID
        JOIN Included i            ON t.TransactionID = i.TransactionID
        JOIN Products p            ON i.ProductID = p.ProductID
        JOIN Processing pr         ON t.TransactionID = pr.TransactionID
        JOIN Store s               ON pr.StoreID = s.StoreID
        LEFT JOIN BrickAndMotor bm ON s.StoreID = bm.StoreID
        LEFT JOIN Web w            ON s.StoreID = w.StoreID
        WHERE c.CustomerID = %s
        ORDER BY t.TransactionDate, t.TransactionID
    """, (cid,))
    rows = cursor.fetchall()
    if not rows:
        print("No transaction history found for that customer.")
    else:
        print(f"\nPurchase history for Customer {rows[0][0]} — {rows[0][1]}")
        print(f"{'TxnID':<7} {'Date':<22} {'Pay':<8} {'Product':<40} {'Qty':>4} {'Price':>7} {'Total':>8}")
        print("-" * 102)
        for cid2, name, txid, date, pay, loc, prod, qty, price, total in rows:
            print(f"{txid:<7} {str(date):<22} {pay:<8} {prod[:38]:<40} {qty:>4} ${price:>6.2f} ${total:>7.2f}")
else:
    print("Invalid choice — enter a number 1 through 8.")

Manager Reports
  1 — Top 20 best-selling products at each store
  2 — Top 20 best-selling products in each state
  3 — Top 5 stores with most sales this year
  4 — Stores where Pork Belly outsells Chicken (Coke vs Pepsi equivalent)
  5 — Top 3 categories bought together with Rice (like 'bought with milk')
  6 — Monthly sales breakdown by store
  7 — Average transaction value per store
  8 — Customer purchase history (enter a Customer ID)



Enter report number (1-8):  1



Store   Location                                 Rank   Product                                   Units    Revenue
-------------------------------------------------------------------------------------------------------------------
1       1235 Highland Ave, National City, CA 9   1      Ligo Sardines in Tomato Sauce 5.5oz         105 $   114.45
1       1235 Highland Ave, National City, CA 9   2      Pork Belly (Liempo) per lb                   61 $   212.89
1       1235 Highland Ave, National City, CA 9   3      Whole Tilapia (Fresh)                        40 $   199.60
1       1235 Highland Ave, National City, CA 9   4      Knorr Sinigang Mix Sampaloc 40g              29 $    43.21
1       1235 Highland Ave, National City, CA 9   5      Chicken Leg Quarters per lb                  25 $    37.25
1       1235 Highland Ave, National City, CA 9   6      Datu Puti Soy Sauce 1L                       24 $    71.76
1       1235 Highland Ave, National City, CA 9   7      Lucky Me Pancit Canton

## Cell 8
Search any value across every column in the database/find any value across all tables.

In [19]:
search_item = input("Enter what to search for: ").strip()

cursor.execute("""
    SELECT DISTINCT table_schema
    FROM information_schema.columns
    WHERE table_schema NOT IN
    ('mysql','information_schema','performance_schema','sys')
""")
schemas = cursor.fetchall()

try:
    numeric_val = int(search_item)
    is_numeric = True
except ValueError:
    is_numeric = False

found_any = False

for (schema,) in schemas:
    cursor.execute("""
        SELECT table_name, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = %s
    """, (schema,))
    columns = cursor.fetchall()

    for table_name, column_name, data_type in columns:

        if data_type in ('varchar', 'char', 'text', 'mediumtext', 'longtext'):
            query = f"""
                SELECT `{column_name}`
                FROM `{schema}`.`{table_name}`
                WHERE `{column_name}` LIKE %s
            """
            cursor.execute(query, ('%' + search_item + '%',))
            for (val,) in cursor.fetchall():
                print(f"{schema}  {table_name}  {column_name}  {val}")
                found_any = True

        elif is_numeric and data_type in ('int', 'bigint', 'smallint', 'tinyint',
                                          'decimal', 'float', 'double'):
            query = f"""
                SELECT `{column_name}`
                FROM `{schema}`.`{table_name}`
                WHERE `{column_name}` = %s
            """
            cursor.execute(query, (numeric_val,))
            for (val,) in cursor.fetchall():        # <-- fixed: was row[0] in original
                print(f"{schema}  {table_name}  {column_name}  {val}")
                found_any = True

if not found_any:
    print(f"No results found for '{search_item}'.")

Enter what to search for:  fish


seafoodcity  Category  SubCategory  Fresh Fish
seafoodcity  Category  SubCategory  Shellfish
seafoodcity  Products  Name  Bangus (Milkfish) Whole
seafoodcity  Products  Name  Whole Catfish
seafoodcity  Seafood  SubCategory  Fresh Fish
seafoodcity  Seafood  SubCategory  Shellfish


---
## Cell 8 — Close Connection
Always run this when you are done to cleanly close the MySQL connection.

In [ ]:
# ── CLOSE CONNECTION ──────────────────────────────────────────
cursor.close()
cnx.close()
print("Connection closed.")